### STEP : PICRUST2 Analysis



#### Example

- [PICRUST2 tutorial](https://github.com/picrust/picrust2/wiki/q2-picrust2-Tutorial)
- [Limitations](https://github.com/picrust/picrust2/wiki/Key-Limitations)


#### Methods
- [composition](https://docs.qiime2.org/2022.8/plugins/available/composition/)

## Setup and settings

In [ ]:
# Importing packages
import os
import pandas as pd
from qiime2 import Artifact
from qiime2 import Visualization
from qiime2 import Metadata

import biom
from biom.util import biom_open

from qiime2.plugins.diversity.pipelines import core_metrics
from qiime2.plugins.feature_table.pipelines import summarize
from qiime2.plugins.feature_table.methods import filter_samples
from qiime2.plugins.feature_table.methods import filter_seqs

%matplotlib inline

### Receiving the parameters

The following cell can receive parameters using the [papermill](https://papermill.readthedocs.io/en/latest/) tool.

In [ ]:
metadata_file = 'metadata.tsv'
base_dir = os.path.join('/', 'home', 'user', 'test-folder')
experiment_name = 'experiment-test'
class_col = 'group-id'
replace_files = False

In [ ]:
# Parameters
experiment_name = "experiment-test"
base_dir = "/mnt/data/qiime2-experiments"
manifest_file = "manifest.csv"
metadata_file = "metadata.tsv"
class_col = "group"
classifier_file = "qiime2-classifiers/silva-138.2-v3v4-341f-806r-nb-classifier.qza"
read_layout = "paired-end"
replace_files = False
phred = 20
trunc_f = 0
trunc_r = 0
overlap = 12
threads = 8
trim = {"overlap": 8, "forward_primer": "CCTACGGGRSGCAGCAG", "reverse_primer": "GGACTACHVGGGTWTCTAAT"}

In [ ]:
experiment_folder = os.path.abspath(os.path.join(base_dir, 'experiments', experiment_name))
img_folder = os.path.abspath(os.path.join(experiment_folder, 'imgs'))

### Defining names, paths and flags

In [ ]:
# QIIME2 Artifacts folder
qiime_folder = os.path.join(experiment_folder, 'qiime-artifacts')

# Input - Artifacts
table_path = os.path.join(qiime_folder, 'dada2-tabs.qza')
taxonomy_path = os.path.join(qiime_folder, 'metatax.qza')

# ANCOM folder
ancom_folder = os.path.abspath(os.path.join(experiment_folder, 'ancom'))

# Create path if it not exist
if not os.path.isdir(ancom_folder):
    os.makedirs(ancom_folder)
    print(f'New ANCOM artifacts folder path created: {ancom_folder}')

In [ ]:
if not os.path.isfile(table_path):
    raise FileNotFoundError(f"The file '{table_path}' does not exist.")

if not os.path.isfile(taxonomy_path):
    raise FileNotFoundError(f"The file '{taxonomy_path}' does not exist.")

if not os.access(table_path, os.R_OK):
    raise FileNotFoundError(f"The file '{table_path}' exists but cannot be read (permission denied).")

if not os.access(taxonomy_path, os.R_OK):
    raise FileNotFoundError(f"The file '{taxonomy_path}' exists but cannot be read (permission denied).")

## Step execution

### Export files

In [ ]:
print("Exporting feature table...")
!qiime tools export \
    --input-path "{table_path}" \
    --output-path "{ancom_folder}"

In [ ]:
print("Exporting taxonomy...")
!qiime tools export \
    --input-path "{taxonomy_path}" \
    --output-path "{ancom_folder}"

In [ ]:
print("Converting feature table from BIOM to TSV...")

in_path = os.path.join(ancom_folder, 'feature-table.biom')
out_path = os.path.join(ancom_folder, 'feature-table.tsv')

!biom convert \
    -i "{in_path}" \
    -o "{out_path}.tmp" \
    --to-tsv

# Remove BIOM comment line, preserving the actual header
!tail -n +2 "{out_path}.tmp" \
    > "{out_path}"

!rm "{out_path}.tmp"

In [ ]:
%%bash
if docker image inspect ancombc2:latest > /dev/null 2>&1; then
    echo "Image exists locally"
else
    echo "Image does not exist locally"

    # Download the ANCOM-BC2 Docker image if it does not exist locally
    !wget -O Dockerfile.ancombc2 https://raw.githubusercontent.com/lauromoraes/microbiom/main/src/Dockerfile.ancombc2

    # Install the ANCOM-BC2 Docker image
    !docker build -t ancombc2:latest -f Dockerfile.ancombc2 .
fi

In [ ]:
# Copy metadata file to ancom folder
!cp {metadata_file} {ancom_folder}

In [ ]:
# Execute the ANCOM-BC2 pipeline using Docker
cmd = f"""Rscript ancombc2.R \
    --input-dir . \
    --output-dir ./output \
    --group-var menopausal-status \
    --reference-level premenopause \
    --tax-level Genus \
    --prevalence-cutoff 0.10 \
    --library-cutoff 1000 \
    --pseudo-sens true \
    --cores 8
"""
!docker run --rm --workdir /data -v {ancom_folder}:/data ancombc2:latest /bin/bash -c "{cmd}"